# Исследование данных для рекомендательной системы книг

## 0. Постановка исследования

### 0.1. Постановка задачи

**Цель проекта**: разработать рекомендательную систему книг и сравнить содержательный, коллаборативный и гибридный подходы для формирования персональных рекомендаций.

Содержательный подход использует описания и характеристики книг. Коллаборативный учитывает пользовательские оценки. Гибридный подход объединяет информацию о книгах и взаимодействиях пользователей.

В исследовании необходимо проверить качество исходных данных, подготовить связанные таблицы книг и взаимодействий, определить протокол оценки и сравнить рекомендательные модели.

### 0.2. Источники данных

Используются три исходных файла из двух открытых источников:

1. `datasets1.parquet`: каталог книг из набора
   [euclaise/goodreads_100k](https://huggingface.co/datasets/euclaise/goodreads_100k).
2. `datasets2.csv`: метаданные произведений из набора
   [Pauleera/Goodreads-Book-Reviews](https://huggingface.co/datasets/Pauleera/Goodreads-Book-Reviews).
3. `reviews.csv`: пользовательские отзывы и оценки из того же источника.

Названия полей, типы данных и связи между таблицами будут проверены
после загрузки.

### 0.3. План исследования данных

1. Загрузка и первичный обзор таблиц.
2. Проверка идентификаторов, пропусков и дубликатов.
3. Анализ описаний и характеристик книг.
4. Анализ пользовательских оценок и активности.
5. Проверка связей между источниками.
6. Обоснование очистки и объединения данных.

## 1. Подготовка среды и предварительный обзор

### 1.1. Загрузка библиотек

При отсутствии необходимых библиотек в окружении проекта для правильной работы исследования рекомендуется выполнение следующей ячейки:

In [1]:
%pip install pandas pyarrow matplotlib seaborn

Note: you may need to restart the kernel to use updated packages.


`%pip` устанавливает необходимые пакеты в окружение текущего ядра. Если после установки появляется просьба перезапуска ядра, перезапустите его перед выполнением следующих ячеек.

### 1.2. Подготовка среды

Подключим библиотеки, настроим отображение таблиц и зафиксируем версии основных пакетов.

In [2]:
import sys
from pathlib import Path

import pandas as pd
import matplotlib
import pyarrow
import pyarrow.parquet as pq
import matplotlib.pyplot as plt
import seaborn as sns

RANDOM_STATE = 42

pd.set_option("display.max_columns", 50)
pd.set_option("display.max_colwidth", 120)

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 5)

versions = pd.Series(
    {
        "Python": sys.version.split()[0],
        "pandas": pd.__version__,
        "seaborn": sns.__version__,
        "pyarrow": pyarrow.__version__,
        "matplotlib": matplotlib.__version__,
    },
    name="version",
)

versions.to_frame()

,version
Python,3.13.2
pandas,3.0.5
seaborn,0.13.2
pyarrow,25.0.1
matplotlib,3.11.1


### 1.3. Расположение исходных файлов

Проверим наличие файлов и их размеры перед загрузкой. Исходные данные остаются без изменений.

In [3]:
current_dir = Path.cwd()

if (current_dir / "data" / "raw").is_dir():
    project_dir = current_dir
elif (current_dir.parent / "data" / "raw").is_dir():
    project_dir = current_dir.parent
else:
    raise FileNotFoundError(
        "Не найдена папка data/raw. "
        "Проверьте структуру проекта и рабочую папку ноутбука."
    )

raw_dir = project_dir / "data" / "raw"

paths = {
    "books_texts": raw_dir / "datasets1.parquet",
    "books_metadata": raw_dir / "datasets2.csv",
    "reviews": raw_dir / "reviews.csv",
}

file_summary = pd.DataFrame(
    [
        {
            "Таблица": name,
            "Файл": path.name,
            "Наличие": path.is_file(),
            "Размер, МиБ": (
                round(path.stat().st_size / 1024**2, 2)
                if path.is_file()
                else None
            ),
        }
        for name, path in paths.items()
    ]
)

file_summary

,Таблица,Файл,Наличие,"Размер, МиБ"
0,books_texts,datasets1.parquet,True,66.39
1,books_metadata,datasets2.csv,True,15.44
2,reviews,reviews.csv,True,1317.24


In [4]:
missing_files = [
    path.name
    for path in paths.values()
    if not path.is_file()
]

if missing_files:
    raise FileNotFoundError(
        "Не найдены исходные файлы: " + ", ".join(missing_files)
    )

print("Все исходные файлы найдены.")

Все исходные файлы найдены.


### 1.4. Предварительный анализ структуры файлов

Проверим схему Parquet и названия столбцов CSV перед полной загрузкой.

На этапе предварительного просмотра значения CSV читаются как строки.
Это позволяет сохранить исходное представление идентификаторов,
включая ведущие нули в ISBN. Числовые поля и даты будут преобразованы
отдельно после проверки их содержимого.

In [5]:
parquet_file = pq.ParquetFile(paths["books_texts"])

print(
    "Число строк в первой таблице:",
    f"{parquet_file.metadata.num_rows:,}".replace(",", " "),
)

parquet_schema = pd.DataFrame(
    [
        {
            "Столбец": field.name,
            "Тип в файле": str(field.type),
        }
        for field in parquet_file.schema_arrow
    ]
)

parquet_schema

Число строк в первой таблице: 100 000


,Столбец,Тип в файле
0,author,string
1,desc,string
2,genre,string
3,isbn,string
4,link,string
5,pages,int64
6,rating,double
7,reviews,int64
8,title,string
9,totalratings,int64


Здесь мы читаем метаданные файла, а не загружаем всю таблицу в память.

Важно проверить тип ISBN в самом Parquet: если ведущие нули были потеряны ещё при создании файла, простое преобразование в строку их не восстановит.

In [6]:
metadata_preview = pd.read_csv(
    paths["books_metadata"],
    nrows=5,
    dtype="string",
    keep_default_na=False,
)

reviews_preview = pd.read_csv(
    paths["reviews"],
    nrows=5,
    dtype="string",
    keep_default_na=False,
)

print("Столбцы таблицы метаданных:")
print(metadata_preview.columns.tolist())

print("\nСтолбцы таблицы отзывов:")
print(reviews_preview.columns.tolist())

Столбцы таблицы метаданных:
['work_id', 'isbn', 'isbn13', 'original_title', 'author', 'original_publication_year', 'num_pages', 'description', 'genres', 'image_url', 'reviews_count', 'text_reviews_count', '5_star_ratings', '4_star_ratings', '3_star_ratings', '2_star_ratings', '1_star_ratings', 'ratings_count', 'avg_rating', 'similar_books']

Столбцы таблицы отзывов:
['review_id', 'user_id', 'work_id', 'started_at', 'read_at', 'date_added', 'rating', 'review_text', 'n_votes', 'n_comments']


`keep_default_na=False` пока сохраняет исходные текстовые обозначения пропусков. Например, строка `"NA"` не будет автоматически превращена в отсутствующее значение. Правила обработки пропусков зададим после знакомства с содержимым.

In [7]:
metadata_preview

,work_id,isbn,isbn13,original_title,author,original_publication_year,num_pages,description,genres,image_url,reviews_count,text_reviews_count,5_star_ratings,4_star_ratings,3_star_ratings,2_star_ratings,1_star_ratings,ratings_count,avg_rating,similar_books
0,2919130,1416534601,9781416534600,Nocturnes,John Connolly,2004,,,"fiction, fantasy, paranormal, mystery, thriller, crime, young-adult",https://s.gr-assets.com/assets/nophoto/book/111x148-bcc042a9c91a29c1d680899eff700a03.png,8820,338,1118,1601,1029,190,58,3996,3.9,
1,52087333,,,Draw Play,Tia Lewis,2016,,Jake:\nI can't believe my coach assigned me a tutor. I'm all that on the field and between the sheets -- who cares a...,"romance, fiction",https://s.gr-assets.com/assets/nophoto/book/111x148-bcc042a9c91a29c1d680899eff700a03.png,2482,204,204,353,274,77,29,937,3.7,
2,1649583,1416505520,9781416505525,Citizen of the Galaxy,Robert A. Heinlein,1957,,"In a distant galaxy, the atrocity of slavery was alive and well, and young Thorby was just another orphaned boy sold...","fiction, young-adult, fantasy, paranormal, children",https://s.gr-assets.com/assets/nophoto/book/111x148-bcc042a9c91a29c1d680899eff700a03.png,16506,447,3539,4351,2863,444,53,11250,4,
3,688299,0060541830,9780060541835,Congo,Michael Crichton,1980,,"Deep in the African rain forest, near the legendary ruins of the Lost City of Zinj, an expedition of eight American ...","fiction, mystery, thriller, crime, fantasy, paranormal",https://s.gr-assets.com/assets/nophoto/book/111x148-bcc042a9c91a29c1d680899eff700a03.png,170916,1633,25081,45775,48505,14001,2926,136288,3.6,
4,3464264,0451528824,9780451528827,Anne of Green Gables,L.M. Montgomery,1908,,"Everyone's favorite redhead, the spunky Anne Shirley, begins her adventures at Green Gables, a farm outside Avonlea,...","fiction, young-adult, children, history, historical fiction, biography, romance",https://s.gr-assets.com/assets/nophoto/book/111x148-bcc042a9c91a29c1d680899eff700a03.png,743392,14586,272952,161856,81578,19933,9099,545418,4.2,


In [8]:
reviews_preview

,review_id,user_id,work_id,started_at,read_at,date_added,rating,review_text,n_votes,n_comments
0,fa7a00c01296e3b2b2e857d79c51ea77,3693bb4f1062b659a354848cf11ca313,6128277,,,2013-12-21 00:00:00.000,5,"Fire is half-human and half monster. Monsters in these lands are beautiful, very bright and colorful and usually hav...",0,0
1,de0f7c8d15e247443e51969becf2878e,3693bb4f1062b659a354848cf11ca313,3270810,,,2013-12-21 00:00:00.000,5,"Katsa is a graceling - blessed with an ability (in her case, ability to kill easily) that marks her as different. He...",0,0
2,e79b49504ef58b2defcdc8b79e2ec3fb,3693bb4f1062b659a354848cf11ca313,4768235,,,2013-12-19 00:00:00.000,5,"This is a fun, light-hearted read. Tammy Jo is having a bad week, a really bad week and despite her lack of magical ...",0,0
3,495c1210a9dbd819cbf7717dfb8b217f,3693bb4f1062b659a354848cf11ca313,6264661,,,2013-12-16 00:00:00.000,5,"Cassel, was a character that I was looking forward to and grew on me as the book grew to and end. He starts out as a...",0,0
4,54fcd8963c9dd56485d2a4ad152699c2,3693bb4f1062b659a354848cf11ca313,3429537,,,2013-12-16 00:00:00.000,5,This is kind of one of those books that I saw getting popular back with it first came out; many good reviews were po...,0,0


In [9]:
first_batch = next(
    parquet_file.iter_batches(batch_size=5),
    None,
)

if first_batch is not None:
    books_preview = first_batch.to_pandas()
else:
    books_preview = pd.DataFrame(
        columns=parquet_file.schema_arrow.names
    )

books_preview

,author,desc,genre,isbn,link,pages,rating,reviews,title,totalratings
0,Laurence M. Hauptman,Reveals that several hundred thousand Indians were affected by the Civil War and that twenty thousand Indians enlist...,"History,Military History,Civil War,American History,American Civil War,Nonfiction,North American Hi...,American Hist...",002914180X,https://goodreads.com/book/show/1001053.Between_Two_Fires,0,3.52,5,Between Two Fires: American Indians in the Civil War,33
1,"Charlotte Fiell,Emmanuelle Dirix",Fashion Sourcebook - 1920s is the first book in a brand-new series by Fiell Publishing that documents comprehensivel...,"Couture,Fashion,Historical,Art,Nonfiction",1906863482,https://goodreads.com/book/show/10010552-fashion-sourcebook-1920s,576,4.51,6,Fashion Sourcebook 1920s,41
2,Andy Anderson,"The seminal history and analysis of the Hungarian Revolution and the workers' councils, perhaps the single most impo...","Politics,History",948984147,https://goodreads.com/book/show/1001077.Hungary_56,124,4.15,2,Hungary 56,26
3,Carlotta R. Anderson,"""All-American Anarchist"" chronicles the life and work of Joseph A. Labadie (1850-1933), Detroit's prominent labor or...","Labor,History",814327079,https://goodreads.com/book/show/1001079.All_American_Anarchist,324,3.83,1,All-American Anarchist: Joseph A. Labadie and the Labor Movement,6
4,Jean Leveille,"Aujourdâ€™hui, lâ€™oiseau nous invite Ã sa table, table surprenante par sa diversitÃ© et son originalitÃ©. Tous ini...",NaN,2761920813,https://goodreads.com/book/show/10010880-les-oiseaux-gourmands,177,4.00,1,Les oiseaux gourmands,1


### 1.5. Результаты предварительного просмотра

Все три исходных файла найдены и доступны для чтения. Первый каталог
содержит 100 000 записей и 10 столбцов. Вторая таблица содержит
20 столбцов с метаданными произведений, третья - 10 столбцов
с пользовательскими отзывами.

Описания книг присутствуют в обоих каталогах. Таблицы метаданных
и отзывов содержат поле work_id, которое необходимо проверить
перед использованием для объединения. Для сопоставления двух
каталогов доступны ISBN.

В просмотренных записях обнаружены короткие ISBN, пример нарушенной
кодировки, нулевое число страниц и ссылки на заглушки обложек.
Распространённость этих особенностей требует проверки на полных
таблицах.

Размер файла отзывов составляет около 1,29 ГиБ. Для первоначального
анализа взаимодействий будут загружены все записи, но без текстов
отзывов. Исходный файл остаётся без изменений.

## 2. Загрузка и первичная проверка данных

### 2.1. Загрузка каталогов книг

Загрузим оба каталога целиком. В CSV сохраним исходное строковое
представление значений. Преобразование числовых полей выполним
после проверки содержимого.

In [10]:
books_texts = pd.read_parquet(paths["books_texts"])

books_metadata = pd.read_csv(
    paths["books_metadata"],
    dtype="string",
    keep_default_na=False,
)

print("Первый каталог:", books_texts.shape)
print("Метаданные произведений:", books_metadata.shape)

Первый каталог: (100000, 10)
Метаданные произведений: (13525, 20)


### 2.2. Загрузка пользовательских взаимодействий

Загрузим идентификаторы, оценки, даты и счётчики реакций.
Тексты отзывов пока не включаем в рабочую таблицу: они не нужны
для первоначального анализа структуры взаимодействий.

Чтение ограничивается столбцами, но не количеством записей.

In [11]:
review_columns = [
    "review_id",
    "user_id",
    "work_id",
    "started_at",
    "read_at",
    "date_added",
    "rating",
    "n_votes",
    "n_comments",
]

reviews = pd.read_csv(
    paths["reviews"],
    usecols=review_columns,
    dtype="string",
    keep_default_na=False,
)

print("Таблица взаимодействий:", reviews.shape)

Таблица взаимодействий: (1143887, 9)


Чтение всё равно требует обработки большого CSV, поэтому эта ячейка может выполняться заметно дольше предыдущей. Мы сокращаем прежде всего объём данных в оперативной памяти.

### 2.3. Размеры таблиц и использование памяти

In [12]:
tables = {
    "books_texts": books_texts,
    "books_metadata": books_metadata,
    "reviews": reviews,
}

table_summary = pd.DataFrame(
    [
        {
            "Таблица": name,
            "Строки": len(df),
            "Столбцы": df.shape[1],
            "Память, МиБ": round(
                df.memory_usage(deep=True).sum() / 1024**2,
                2,
            ),
        }
        for name, df in tables.items()
    ]
)

table_summary

,Таблица,Строки,Столбцы,"Память, МиБ"
0,books_texts,100000,10,109.08
1,books_metadata,13525,20,17.04
2,reviews,1143887,9,225.09


Размер файла и занимаемая таблицей память могут существенно различаться. Здесь измеряем именно память загруженного DataFrame.

### 2.4. Пропуски и пустые значения

Для каждого столбца отдельно подсчитаем системные пропуски
и строки, которые становятся пустыми после удаления крайних пробелов.

Проверка не изменяет исходные таблицы. Текстовые обозначения
вроде `"null"` и `"N/A"` на данном этапе автоматически не заменяются.

In [13]:
def missing_report(df):
    rows = []

    for column in df.columns:
        values = df[column]
        missing_count = int(values.isna().sum())

        empty_count = 0

        if (
            pd.api.types.is_object_dtype(values.dtype)
            or pd.api.types.is_string_dtype(values.dtype)
        ):
            empty_count = int(
                values.astype("string")
                .str.strip()
                .eq("")
                .fillna(False)
                .sum()
            )

        total_missing = missing_count + empty_count

        rows.append(
            {
                "Столбец": column,
                "Тип": str(values.dtype),
                "Системные пропуски": missing_count,
                "Пустые строки": empty_count,
                "Всего отсутствует": total_missing,
                "Доля, %": (
                    round(total_missing / len(df) * 100, 2)
                    if len(df)
                    else float("nan")
                ),
            }
        )

    return (
        pd.DataFrame(rows)
        .sort_values("Доля, %", ascending=False)
        .reset_index(drop=True)
    )

In [14]:
missing_books_texts = missing_report(books_texts)
missing_books_texts

,Столбец,Тип,Системные пропуски,Пустые строки,Всего отсутствует,"Доля, %"
0,isbn,str,14482,0,14482,14.48
1,genre,str,10467,0,10467,10.47
2,desc,str,6772,0,6772,6.77
3,author,str,0,0,0,0.00
4,link,str,0,0,0,0.00
5,pages,int64,0,0,0,0.00
6,rating,float64,0,0,0,0.00
7,reviews,int64,0,0,0,0.00
8,title,str,1,0,1,0.00
9,totalratings,int64,0,0,0,0.00


In [15]:
missing_books_metadata = missing_report(books_metadata)
missing_books_metadata

,Столбец,Тип,Системные пропуски,Пустые строки,Всего отсутствует,"Доля, %"
0,similar_books,string,0,2515,2515,18.60
1,isbn,string,0,2051,2051,15.16
2,isbn13,string,0,1661,1661,12.28
3,num_pages,string,0,730,730,5.40
4,description,string,0,169,169,1.25
5,original_publication_year,string,0,18,18,0.13
6,original_title,string,0,0,0,0.00
7,work_id,string,0,0,0,0.00
8,genres,string,0,0,0,0.00
9,image_url,string,0,0,0,0.00


In [16]:
missing_reviews = missing_report(reviews)
missing_reviews

,Столбец,Тип,Системные пропуски,Пустые строки,Всего отсутствует,"Доля, %"
0,started_at,string,0,347495,347495,30.38
1,read_at,string,0,112122,112122,9.80
2,rating,string,0,37318,37318,3.26
3,review_id,string,0,0,0,0.00
4,work_id,string,0,0,0,0.00
5,user_id,string,0,0,0,0.00
6,date_added,string,0,0,0,0.00
7,n_votes,string,0,0,0,0.00
8,n_comments,string,0,0,0,0.00


### 2.5. Первичная проверка идентификаторов

Проверим заполненность и повторяемость ключевых полей.

При проверке временно убираем пробелы по краям идентификаторов.
Исходные столбцы не изменяем. Повторы пока не удаляем:
их смысл зависит от назначения таблицы.

In [17]:
def key_report(df, columns):
    rows = []

    for column in columns:
        values = df[column].astype("string").str.strip()

        missing_mask = values.isna() | values.eq("").fillna(False)
        valid_values = values[~missing_mask]

        rows.append(
            {
                "Поле": column,
                "Отсутствует": int(missing_mask.sum()),
                "Уникальных непустых": valid_values.nunique(),
                "Строк сверх первого вхождения": int(
                    valid_values.duplicated().sum()
                ),
                "Строк в повторяющихся группах": int(
                    valid_values.duplicated(keep=False).sum()
                ),
            }
        )

    return pd.DataFrame(rows)

In [18]:
key_report(books_texts, ["isbn", "link"])

,Поле,Отсутствует,Уникальных непустых,Строк сверх первого вхождения,Строк в повторяющихся группах
0,isbn,14482,85518,0,0
1,link,0,100000,0,0


In [19]:
key_report(books_metadata, ["work_id", "isbn", "isbn13"])

,Поле,Отсутствует,Уникальных непустых,Строк сверх первого вхождения,Строк в повторяющихся группах
0,work_id,0,13525,0,0
1,isbn,2051,11474,0,0
2,isbn13,1661,11864,0,0


In [20]:
key_report(reviews, ["review_id", "user_id", "work_id"])

,Поле,Отсутствует,Уникальных непустых,Строк сверх первого вхождения,Строк в повторяющихся группах
0,review_id,0,1143887,0,0
1,user_id,0,18800,1125087,1143417
2,work_id,0,13525,1130362,1143887


Разница между двумя последними показателями такая: для значений А, А, А, Б число строк сверх первого вхождения равно 2, а число строк в повторяющейся группе равно 3.

### 2.6. Исходные значения пользовательских оценок

Проверим частоты исходных значений `rating` перед преобразованием
в числовой формат. Нулевые, пустые и нестандартные значения
пока сохраняются.

In [21]:
rating_counts = (
    reviews["rating"]
    .value_counts(dropna=False)
    .rename_axis("Исходное значение")
    .reset_index(name="Количество")
)

rating_counts["Доля, %"] = (
    rating_counts["Количество"] / len(reviews) * 100
).round(2)

rating_counts

,Исходное значение,Количество,"Доля, %"
0,4,396722,34.68
1,5,336246,29.4
2,3,242109,21.17
3,2,94030,8.22
4,1,37462,3.27
5,,37318,3.26


### 2.7. Выводы по загрузке и первичной проверке данных

**Объём данных**

Загружены три таблицы: первый каталог содержит 100 000 записей
и 10 столбцов, таблица метаданных содержит 13 525 записей
и 20 столбцов, таблица пользовательских взаимодействий содержит
1 143 887 записей и 9 выбранных столбцов.

Из файла отзывов загружены все записи, но исключён столбец
review_text. Для первоначальной проверки взаимодействий достаточно
идентификаторов, оценок, дат и счётчиков реакций. Тексты отзывов
сохранены в исходном файле и при необходимости могут быть
исследованы отдельно.

Объём памяти, занимаемый тремя DataFrame по результатам
memory_usage(deep=True), составляет около 351,21 МиБ.
Это значение не включает всю память процесса Python и временные
объекты, возникающие при вычислениях.

**Заполненность первого каталога**

В первом каталоге отсутствуют 14 482 ISBN (14,48%),
10 467 значений жанра (10,47%) и 6 772 описания (6,77%).
Также обнаружена одна запись без названия. В остальных полях
системные пропуски и пустые строки не выявлены.

Отсутствие формальных пропусков не гарантирует корректность
значений. В дальнейшем необходимо проверить нулевые значения
числа страниц, качество текстов и возможные текстовые
обозначения отсутствующих данных.

Все 85 518 непустых ISBN уникальны после удаления крайних
пробелов при проверке. Все 100 000 ссылок также заполнены
и уникальны. При этом уникальность ISBN не подтверждает
правильность их формата, а уникальность ссылок не исключает
наличия разных изданий одного произведения.

**Заполненность метаданных произведений**

В таблице метаданных отсутствуют 169 описаний (1,25%).
Это существенно меньшая доля, чем в первом каталоге,
но качество самих текстов ещё не оценивалось.

Поле isbn не заполнено в 2 051 записи (15,16%),
isbn13 в 1 661 записи (12,28%). Эти группы могут пересекаться:
число произведений без обоих идентификаторов необходимо
рассчитать отдельно.

Число страниц отсутствует в 730 записях (5,40%),
год первой публикации в 18 записях (0,13%),
список similar_books в 2 515 записях (18,60%).

Все 13 525 значений work_id заполнены и уникальны.
Повторы среди непустых isbn и isbn13 не обнаружены.
Это позволяет рассматривать work_id как ключ таблицы,
однако соответствие идентификаторов между источниками
требует отдельной проверки.

**Структура пользовательских взаимодействий**

Таблица взаимодействий содержит 18 800 уникальных пользователей
и 13 525 уникальных идентификаторов произведений.
Все 1 143 887 значений review_id заполнены и уникальны.

Повторение user_id и work_id ожидаемо: один пользователь
может взаимодействовать с несколькими произведениями,
а одно произведение может получать отзывы разных пользователей.
Повторные сочетания user_id и work_id на данном этапе
ещё не исследовались.

Число уникальных work_id во взаимодействиях совпадает
с числом произведений в метаданных. Однако совпадение
количеств не доказывает равенство наборов идентификаторов.
Перед объединением необходимо проверить ссылки между таблицами.

**Оценки пользователей**

В поле rating обнаружены значения от 1 до 5 и пустые строки.
Пустая оценка присутствует в 37 318 взаимодействиях (3,26%).

Наиболее распространена оценка 4: 396 722 записи (34,68%).
Оценка 5 встречается в 336 246 записях (29,40%).
Вместе оценки 4 и 5 составляют 64,08% всех взаимодействий
и 66,24% взаимодействий с заполненной оценкой.

Таким образом, в наблюдаемых данных преобладают высокие оценки.
Это необходимо учитывать при определении положительных
взаимодействий и интерпретации качества рекомендаций.
Распределение характеризует представленные в датасете отзывы
и не описывает отношение пользователей ко всем книгам каталога.

Пустые оценки пока сохраняются. Они не трактуются как нулевые
или отрицательные: запись подтверждает наличие отзыва,
но не содержит явной числовой оценки предпочтения.

**Даты взаимодействий**

Дата начала чтения started_at отсутствует в 347 495 записях
(30,38%), дата завершения чтения read_at в 112 122 записях
(9,80%). Поле date_added заполнено во всех записях.

Полнота date_added позволяет рассматривать его для временного
анализа. Однако сначала необходимо проверить преобразование
значений в даты, их диапазон и смысл. Дата добавления записи
не обязательно совпадает с датой чтения или выставления оценки.

**Дальнейшие действия**

На следующем этапе необходимо проверить:

1. Текстовые обозначения пропусков и корректность числовых полей.
2. Форматы и контрольные цифры ISBN.
3. Повторные пары пользователя и произведения.
4. Соответствие work_id между метаданными и взаимодействиями.
5. Форматы, диапазоны и заполненность преобразованных дат.

Объединение, удаление записей и заполнение пропусков пока
не выполнялись. Правила обработки будут определены
по результатам дополнительных проверок.